
## DDL Gold: pf.gold.dim_task  (SCD Type 1)
## Dimension de tarea del modelo (pipeline_tag).

In [0]:
%sql

DROP TABLE IF EXISTS pf.gold.dim_task;

CREATE TABLE IF NOT EXISTS pf.gold.dim_task (
    task_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'PK - Surrogate Key',
    pipeline_tag STRING NOT NULL COMMENT 'BK - tag de tarea',
    descripcion STRING COMMENT 'Descripcion funcional',
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP COMMENT 'UTC',
    PRIMARY KEY (task_id),
    CONSTRAINT uniq_dim_task UNIQUE (pipeline_tag)
)
USING DELTA
TBLPROPERTIES (
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults = 'supported'
)
COMMENT 'Dimension Tarea - SCD Type 1';

In [0]:
%sql

-- Carga/actualizacion SCD1
MERGE INTO pf.gold.dim_task AS t
USING (
    SELECT pipeline_tag AS pipeline_tag,
           CASE pipeline_tag
               WHEN 'text-generation' THEN 'Generacion de texto (LLM)'
               WHEN 'text-classification' THEN 'Clasificacion de texto'
               WHEN 'sentence-similarity' THEN 'Similitud de frases / embeddings'
               WHEN 'text-embedding' THEN 'Embeddings de texto'
               WHEN 'fill-mask' THEN 'Enmascarado de texto'
               WHEN 'token-classification' THEN 'NER / etiquetado de tokens'
               WHEN 'translation' THEN 'Traduccion'
               WHEN 'summarization' THEN 'Resumen'
               WHEN 'question-answering' THEN 'Pregunta-respuesta'
               WHEN 'automatic-speech-recognition' THEN 'Reconocimiento de voz'
               WHEN 'image-classification' THEN 'Clasificacion de imagenes'
               ELSE 'Otra'
           END AS descripcion
    FROM pf.silver.modelos
    WHERE pipeline_tag IS NOT NULL
    GROUP BY pipeline_tag
) AS s
ON t.pipeline_tag = s.pipeline_tag
WHEN MATCHED THEN
    UPDATE SET t.descripcion = s.descripcion
WHEN NOT MATCHED THEN
    INSERT (pipeline_tag, descripcion, _createdAt)
    VALUES (s.pipeline_tag, s.descripcion, CURRENT_TIMESTAMP());

In [0]:
%sql

SELECT 
    task_id, 
    pipeline_tag, 
    descripcion 
FROM pf.gold.dim_task 
ORDER BY pipeline_tag;